# Skill 38: Diffusion Forward Process

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Corrupt data with Gaussian noise under a variance schedule and jump to any timestep in closed form.

## Objectives
- Create beta and cumulative alpha schedules.
- Sample $x_t$ directly from $x_0$ without simulating every step.
- Observe an image becoming increasingly noisy across timesteps.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \bar\alpha_t=\prod_{s\le t}(1-\beta_s)$$
Linear schedule: $\beta$ from 1e-4 to 0.02 over T=1000; $\bar\alpha=0.5$ near t=260.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import matplotlib.pyplot as plt

T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def add_noise(x0, t):
    # t is a zero-based integer timestep.
    eps = torch.randn_like(x0)
    a = alpha_bar[t].to(x0.device)
    xt = torch.sqrt(a) * x0 + torch.sqrt(1 - a) * eps
    return xt, eps

# A simple synthetic "image" with values scaled to [-1, 1].
x0 = torch.zeros(1, 1, 32, 32)
x0[:, :, 8:24, 8:24] = 1.0
timesteps = [0, 100, 300, 600, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(12, 3))
for ax, t in zip(axes, timesteps):
    xt, _ = add_noise(x0, t)
    ax.imshow(xt[0, 0].numpy(), cmap="gray", vmin=-2, vmax=2)
    ax.set_title(f"t={t}")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("alpha_bar at final step:", alpha_bar[-1].item())


## What to check
**Expected result:** Noise dominates at late timesteps; `alpha_bar[-1]` is about 4e-5. Note: the image holds values 0 and 1 (the code comment says [-1,1]); real DDPM training rescales to [-1,1].

## Pitfalls
Use `alpha_bar`, not `alphas`. Papers index t from 1, this code from 0. Batched t needs reshaping to (B,1,1,1).